# Recon Analysis

Compare two session files for differences

**Before running:** set `DATA_FOLDER` to the folder containing your
session output CSV files. 

In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

DATA_FOLDER = Path(r"C:\Git\CandleStateSessionAnalysis\data\recon")
OUTPUT_FOLDER = DATA_FOLDER / "output" 

In [ ]:
import parse_sessions

tables = parse_sessions.parse_sessions(DATA_FOLDER)

# from IPython.display import display
# for table_name, df in tables.items():
#     print(f"### {table_name}")
#     display(df.describe())

trade_signals = tables["trade_signals"]
# out_path = OUTPUT_FOLDER / f"trade_signals.csv"
# trade_signals.to_csv(out_path, index=False)

In [ ]:

key_cols = ['Symbol', 'CandleOpen']
compare_cols = ['Close', 'Volume', 'VolumeRatio', 'VWAP']

pivot = trade_signals.pivot_table(
    index=key_cols,
    columns='SourceFile',
    values=compare_cols,
    aggfunc='first'
)

pivot.columns = [f'{val}__{src}' for val, src in pivot.columns]
pivot = pivot.reset_index()

sources = sorted(trade_signals['SourceFile'].unique())
s1, s2 = sources

pivot['Close_diff']  = pivot[f'Close__{s1}']  - pivot[f'Close__{s2}']
pivot['Volume_diff'] = pivot[f'Volume__{s1}'] - pivot[f'Volume__{s2}']

diffs_only = pivot[(pivot['Close_diff'] != 0) | (pivot['Volume_diff'] != 0)]
diffs_only.head(10)

In [ ]:
import matplotlib.pyplot as plt

trade_signals['CandleOpen'] = pd.to_datetime(trade_signals['CandleOpen'])

compare_cols = ['Close', 'Volume', 'VolumeRatio', 'VWAP']
symbols = sorted(trade_signals['Symbol'].unique())
sources = sorted(trade_signals['SourceFile'].unique())
s1, s2 = sources

fig, axes = plt.subplots(len(symbols), len(compare_cols),
                          figsize=(4*len(compare_cols), 3*len(symbols)),
                          sharex='col')

for i, sym in enumerate(symbols):
    sub = trade_signals[trade_signals['Symbol'] == sym]
    for j, col in enumerate(compare_cols):
        ax = axes[i, j]
        for src, style in zip([s1, s2], ['-', '--']):
            s = sub[sub['SourceFile'] == src].sort_values('CandleOpen')
            ax.plot(s['CandleOpen'], s[col], style,
                    label=src.replace('Session_Live__','').replace('.json',''),
                    linewidth=1)
        if i == 0:
            ax.set_title(col)
        if j == 0:
            ax.set_ylabel(sym)
        ax.tick_params(axis='x', rotation=45, labelsize=6)

axes[0, 0].legend(fontsize=6, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# --- Positions recon ---
positions = tables["positions"].copy()

# Match positions across files by Symbol + IndexOpen (entry candle index).
# IDs are random per run and Opened timestamps differ by a few seconds
# between live and file replay, so the candle index is the stable key.
key_cols = ['Symbol', 'IndexOpen']
compare_cols = ['Opened', 'Closed', 'IndexClose', 'Quantity', 'AvgPrice',
                'ClosePrice', 'StopPrice', 'PeakUnrealizedGain', 'RealizedGain']
numeric_cols = ['IndexClose', 'Quantity', 'AvgPrice', 'ClosePrice',
                'StopPrice', 'PeakUnrealizedGain', 'RealizedGain']

# Short labels ("File", "Live") pulled from the filename
positions['Source'] = positions['SourceFile'].str.extract(r'Session_(\w+)__')
sources = sorted(positions['Source'].unique())
s1, s2 = sources

merged = pd.merge(
    positions[positions['Source'] == s1][key_cols + compare_cols],
    positions[positions['Source'] == s2][key_cols + compare_cols],
    on=key_cols, how='outer',
    suffixes=(f'__{s1}', f'__{s2}'), indicator=True,
)

# Flag entries taken in one run but not the other
unmatched = merged[merged['_merge'] != 'both']
if not unmatched.empty:
    print("Positions present in only one file:")
    print(unmatched[key_cols + ['_merge']].to_string(index=False))

for col in numeric_cols:
    merged[f'{col}_diff'] = merged[f'{col}__{s1}'] - merged[f'{col}__{s2}']

# Interleave: value__File, value__Live, value_diff for each column
col_order = key_cols.copy()
for col in compare_cols:
    col_order += [f'{col}__{s1}', f'{col}__{s2}']
    if col in numeric_cols:
        col_order.append(f'{col}_diff')

recon = merged[col_order].sort_values(key_cols).reset_index(drop=True)
recon.T   # transpose: one column per position, easy to eyeball


In [ ]:
# --- Compact positions recon: Live vs File side by side ---
positions = tables["positions"].copy()
positions['Source'] = positions['SourceFile'].str.extract(r'Session_(\w+)__')

view_cols = ['IndexOpen', 'AvgPrice', 'IndexClose', 'ClosePrice', 'Quantity']

# Number positions within each symbol by entry order, so rows still
# line up even if Live and File entered on slightly different candles.
positions['PosNum'] = (positions
    .sort_values('IndexOpen')
    .groupby(['Source', 'Symbol'])
    .cumcount() + 1)

live = positions[positions['Source'] == 'Live'].set_index(['Symbol', 'PosNum'])[view_cols]
file = positions[positions['Source'] == 'File'].set_index(['Symbol', 'PosNum'])[view_cols]

compact = live.join(file, how='outer', lsuffix='_Live', rsuffix='_File').reset_index()

compact = compact.rename(columns={
    'AvgPrice_Live': 'Open$_Live',   'ClosePrice_Live': 'Close$_Live',
    'AvgPrice_File': 'Open$_File',   'ClosePrice_File': 'Close$_File',
    'Quantity_Live': 'Qty_Live',     'Quantity_File': 'Qty_File',
})

# Diffs: Live minus File. PerShare_Slippage is the per-share P&L gap
# between the two runs: (Close - Open)_Live - (Close - Open)_File.
compact['Open$_Diff']        = compact['Open$_Live']  - compact['Open$_File']
compact['Close$_Diff']       = compact['Close$_Live'] - compact['Close$_File']
compact['PerShare_Slippage'] = compact['Close$_Diff'] - compact['Open$_Diff']

compact = compact[['Symbol',
    'IndexOpen_Live', 'Open$_Live', 'IndexClose_Live', 'Close$_Live', 'Qty_Live',
    'IndexOpen_File', 'Open$_File', 'IndexClose_File', 'Close$_File', 'Qty_File',
    'Open$_Diff', 'Close$_Diff', 'PerShare_Slippage']]

compact.round(4)